# Li–hBN × Clifford QC: reproducibility and eigenvector checks

**Evidence:** default is **SYNTHETIC**. This notebook does not re-execute the 20-day QE Hartree–Fock/ACE calculation and does not claim a Li–hBN interacting wavefunction. It reproduces the *verification protocol* from the supplied October 20 Python logbook, with optional original-file checks. Each calculation is labelled by evidence tier.

**Colab:** use Runtime → Run all on CPU. The initial source mode is GITHUB and clones the published feature branch. Set SOURCE_MODE to ZIP if you prefer uploading the offline experiment archive. To use original materials, mount Drive in the configuration cell and set `DATA_MODE="REAL"` after placing the original files there. The private production inputs are **not** bundled in this public GitHub repository. [Methods and limitations](../experiments/lihbn/README.md).

In [ ]:
import os, sys, subprocess, json, platform, hashlib, tempfile, time
from pathlib import Path
import numpy as np

SOURCE_BRANCH = "feat/lihbn-colab-reproduction"  # switch to main after PR merge
SOURCE_MODE = "GITHUB"  # Use ZIP only for offline source uploads
if (Path.cwd() / "lihbn_repro" / "wannier.py").exists():
    PROJECT = Path.cwd().resolve()  # local notebook tests
else:
    PROJECT = Path('/content/colabexercise-lihbn')
    if not (PROJECT / 'lihbn_repro' / 'wannier.py').exists():
        if SOURCE_MODE == 'GITHUB':
            subprocess.run(['git','clone','--depth','1','--branch',SOURCE_BRANCH,
                            'https://github.com/gutama/colabexercise.git',str(PROJECT)],check=True)
        elif SOURCE_MODE == 'ZIP':
            from google.colab import files
            import io, zipfile
            print('Upload the downloaded lihbn_colab_reproduction_package.zip')
            uploaded = files.upload()
            if len(uploaded) != 1:
                raise ValueError('Upload exactly one Li-hBN reproduction ZIP')
            archive_bytes=next(iter(uploaded.values()))
            with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
                for member in z.namelist():
                    if member.startswith('/') or '..' in Path(member).parts:
                        raise ValueError('Unsafe path inside archive')
                PROJECT.mkdir(parents=True,exist_ok=True)
                z.extractall(PROJECT)
        else:
            raise ValueError('SOURCE_MODE must be ZIP or GITHUB')
sys.path.insert(0, str(PROJECT))
from lihbn_repro import make_demo_hr, parse_hr, spin_blocks, one_particle_eigenpairs, sector_crosscheck
from lihbn_repro.qe_audit import audit as audit_qe
from lihbn_repro.validation import compare_eig_csv
try:
    COMMIT = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    COMMIT = 'local-uncommitted'
print('Repository:', PROJECT, 'commit:', COMMIT)
print('Python:', sys.version.split()[0], 'NumPy:',np.__version__)

## 1. Choose evidence tier and upload/mount data

**DEMO** creates deterministic **synthetic** 11×11 spin blocks for testing. **REAL** requires original *two-spin* 97-Wannier `*_hr.dat` (149 R-vectors each). Do not substitute paper table values, generic toy parameters or a fake HF output for physical data. `H_MODE="onsite"` implements the notebook's `H0_up`/`H0_dn` R=0 selection, *not* Fourier `H(k=Gamma)`.

Active-sector electron counts have not been recovered; never guess them from total cell magnetization. Leave `N_UP`/`N_DOWN` as `None` until validated. The small DEMO uses `(1,1)` only to test Slater determinant reconstruction.

In [ ]:
DATA_MODE = 'DEMO'  # 'DEMO' or 'REAL'
H_MODE = 'onsite'  # 'onsite': R=0; 'gamma': Fourier sum over all R
UP_HR = '/content/drive/MyDrive/lihbn/lihbn_97wf_up_hr.dat'
DOWN_HR = '/content/drive/MyDrive/lihbn/lihbn_97wf_down_hr.dat'
N_UP, N_DOWN = None, None  # REAL: fill only from verified active-space occupations
HF_LOG = None  # e.g. '/content/drive/MyDrive/lihbn/lihbn_hf_k3.out'
NSCF_CSV, W90_EIG_UP, W90_EIG_DOWN = None, None, None

if DATA_MODE == 'REAL':
    # In Google Colab: use these two lines if the files are on Google Drive:
    # from google.colab import drive
    # drive.mount('/content/drive')
    if not Path(UP_HR).is_file() or not Path(DOWN_HR).is_file():
        raise FileNotFoundError('REAL mode requires original UP and DOWN *_hr.dat files')
    if (N_UP is None) != (N_DOWN is None):
        raise ValueError('Both N_UP and N_DOWN must be supplied together')
    up_path, down_path = Path(UP_HR), Path(DOWN_HR)
    labels = None
elif DATA_MODE == 'DEMO':
    folder = Path(tempfile.mkdtemp(prefix='lihbn-synthetic-'))
    up_path = make_demo_hr(folder/'SYNTHETIC_up_hr.dat',nw=11,seed=6)
    down_path = make_demo_hr(folder/'SYNTHETIC_down_hr.dat',nw=11,seed=9)
    N_UP, N_DOWN = 1, 1  # synthetic test only, no Li-hBN claim
    labels = list(range(1,12))
else:
    raise ValueError('DATA_MODE must be DEMO or REAL')
print('EVIDENCE:', DATA_MODE, '| mode:', H_MODE)

## 2. Parse Wannier90 `_hr.dat` and verify integrity

The checks enforce **unique `(R,m,n)` coverage**, 149/97 only in REAL mode, `H(R)=H(-R)†`, partner degeneracies, and H(k) Hermiticity. They **do not** verify agreement with direct Wannier gauge `T†ET` or resolve the historical printed-HR Fourier discrepancy.

In [ ]:
expected = (97,149) if DATA_MODE == 'REAL' else (11,3)
up = parse_hr(up_path,expected_wann=expected[0],expected_nrpts=expected[1])
down = parse_hr(down_path,expected_wann=expected[0],expected_nrpts=expected[1])
for tag, obj in [('UP',up),('DOWN',down)]:
    gamma_error = np.max(np.abs(obj.h_at_k()-obj.h_at_k().conj().T))
    print(tag,'SHA256:',obj.sha256,'Wannier:',obj.num_wann,'R:',obj.nrpts,
          'max H(R) symmetry error:',obj.max_hr_hermiticity,
          'Gamma Hermiticity error:',gamma_error)
    assert gamma_error < 1e-7
h_up,h_down = spin_blocks(up,down,mode=H_MODE,**({'up_labels':labels,'down_labels':labels} if labels else {}))
assert h_up.shape == h_down.shape == (11,11)
print('Selected spin blocks:',h_up.shape,h_down.shape,'(eV; spin-blocked order)')

## 3. Obtain one-particle eigenvectors and optionally the quadratic Slater state

The 22-mode spin-blocked model is `diag(h_up,h_down)`. `numpy.linalg.eigh` returns **orbital eigenvectors** as columns. When occupations are given, occupied orbital projectors form a Slater determinant and its 1-RDMs. This is **not** a correlated many-body state; the active-space two-electron tensor and double counting are unresolved in the source notebook.

In [ ]:
res_up = one_particle_eigenpairs(h_up,N_UP)
res_down = one_particle_eigenpairs(h_down,N_DOWN)
for tag, result in [('UP',res_up),('DOWN',res_down)]:
    print(tag,'lowest 5 orbital energies (eV):',np.round(result['energies_ev'][:5],8))
    print('  max eigenvector residual (eV):',result['max_residual_ev'],
          'orthogonality defect:',result['orthogonality_defect'])
    assert result['max_residual_ev'] < 1e-8
if N_UP is not None:
    print('Quadratic Slater energy (eV, NO two-body effects):',
          res_up['slater_energy_ev'] + res_down['slater_energy_ev'])
else:
    print('No many-body electron counts supplied: Slater state intentionally omitted.')

## 4. Bounded independent fixed-sector cross-check

For a **small** fixed `(N_up,N_down)` sector, build the second-quantized `Σ h_pq a†_p a_q` Hamiltonian directly using fermionic bit signs, diagonalize, and independently reconstruct the occupied-orbital Slater amplitudes. Compare energies, residual, and (for a nondegenerate root) squared fidelity. If sector dimension exceeds 500, **skip safely**.

In [ ]:
sector = None
if N_UP is not None:
    sector = sector_crosscheck(h_up,h_down,N_UP,N_DOWN,max_dimension=500)
    print(json.dumps(sector,indent=2))
    if not sector['skipped']:
        assert sector['energy_error_ev'] < 1e-7
        assert sector['slater_residual_ev'] < 1e-7
else:
    print('SKIPPED: original active-space electron populations have not been supplied.')

## 5. Optional real-data evidence checks

These checks **only read** supplied files. A `convergence has been achieved` inner-SCF line does not mean the full ACE/HF job completed; only a final completion marker plus scientific checks can support a completed calculation. `.eig` vs NSCF comparison needs the original CSV with 57,600 rows (144 k-points × 200 bands × 2 spins).

In [ ]:
hf_audit = None
if HF_LOG:
    hf_audit = audit_qe(Path(HF_LOG), processes=False)
    print('QE HF status:',json.dumps({k:hf_audit[k] for k in
        ('job_done','ace_mentioned','counts','last_total_energy_ry','last_ace_fock_energy_ry')},indent=2))
    if not hf_audit['job_done']:
        print('NOT A COMPLETED HF/ACE RESULT.')
else:
    print('HF/ACE log audit skipped; no original log provided.')

spectral_check = None
if all([NSCF_CSV,W90_EIG_UP,W90_EIG_DOWN]):
    spectral_check = compare_eig_csv(NSCF_CSV,W90_EIG_UP,W90_EIG_DOWN)
    print('W90 .eig vs QE NSCF:',json.dumps(spectral_check,indent=2))
else:
    print('NSCF/Wannier .eig comparison skipped; original files not supplied.')

## 6. Export immutable numerical evidence locally (not to public GitHub)

SHA-256 identifies the input bytes; report explicitly distinguishes DEMO from REAL. The NPZ includes eigenvectors, orbital energies and input spin blocks, but never a fabricated 28-qubit correlated state. Store these results privately or copy them to Drive for your supervisor.

In [ ]:
from datetime import datetime,timezone
OUT = Path(tempfile.mkdtemp(prefix='lihbn-verified-'))
arrays={'h_up_ev':h_up,'h_down_ev':h_down,
        'eig_up_ev':res_up['energies_ev'],'eig_down_ev':res_down['energies_ev'],
        'vectors_up':res_up['orbitals'],'vectors_down':res_down['orbitals']}
if N_UP is not None:
    arrays.update(one_rdm_up=res_up['one_rdm'],one_rdm_down=res_down['one_rdm'])
np.savez_compressed(OUT/'onebody_eigenvectors.npz',**arrays)
report={'schema':'lihbn_repro.v1','observed_at_utc':datetime.now(timezone.utc).isoformat(),
        'evidence_tier':'synthetic_demo' if DATA_MODE=='DEMO' else 'original_input_unverified_physics',
        'mode':H_MODE,'source_note':'mylogbook_python_snippets_2026-10-20_v02.tex',
        'repository_commit':COMMIT,'python':sys.version.split()[0],'numpy':np.__version__,
        'up_hr_sha256':up.sha256,'down_hr_sha256':down.sha256,
        'up_labels_1based':labels if labels else list(__import__('lihbn_repro').UP_WANNIER_LABELS),
        'down_labels_1based':labels if labels else list(__import__('lihbn_repro').DOWN_WANNIER_LABELS),
        'onebody_max_residual_ev':max(res_up['max_residual_ev'],res_down['max_residual_ev']),
        'sector':sector,'hf_read_only':({k:hf_audit.get(k) for k in ('job_done','last_total_energy_ry','last_ace_fock_energy_ry')} if hf_audit else None),
        'eig_nscf_validation':spectral_check,
        'limitations':['H(R=0) differs from Fourier Gamma; not a direct DFT gauge validation',
                       'No actual QE Hartree-Fock/ACE calculation was run',
                       'No validated two-body interacting Li-hBN Hamiltonian',
                       'No correlated many-body wavefunction reported']}
(OUT/'evidence.json').write_text(json.dumps(report,indent=2)+'\n')
print('Results kept outside public GitHub:',OUT)
print('Evidence:',report['evidence_tier'])

## 7. Optional `clifford_qc` integration smoke test — synthetic Hubbard dimer

This is **independent** of Li-hBN. It tests whether the existing `clifford_qc` sector eigensolver returns energies and **state vectors** with small residual on a known two-site, two-electron interacting Hubbard model. Install a pinned source revision if requested; don't infer real Li-hBN agreement from this test.

In [ ]:
RUN_CLIFFORD_SMOKE = False  # opt-in; requires network installation
if RUN_CLIFFORD_SMOKE:
    CLIFFORD_COMMIT='bd772c4ec645409db21d742547dbf054e1520757'
    subprocess.run([sys.executable,'-m','pip','install','-q',
        'git+https://github.com/gutama/clifford_qc.git@'+CLIFFORD_COMMIT],check=True)
    from clifford_qc.models import effective_hamiltonian
    from clifford_qc.backends import SectorStatevectorBackend
    toy={'schema':'clifford_qc.effective_hamiltonian.v1','name':'SYNTHETIC Hubbard dimer',
         'energy_unit':'eV','one_body':[[0.,-1.],[-1.,0.]],'onsite_u':[4.,4.],
         'reference_occupied_spin_orbitals':[0,3], 'sector':{'n_electrons':2,'sz':0.}}
    model=effective_hamiltonian(toy)
    backend=SectorStatevectorBackend(model.n,n_electrons=2,sz=0.0)
    energies,states=backend.ground_state(model.hamiltonian,k=1,method='dense')
    H_action=backend.operator(model.hamiltonian)
    residual=np.linalg.norm(H_action.matvec(states[:,0])-energies[0]*states[:,0])
    expected=(4.-np.sqrt(4.**2+16.))/2.
    print({'toy_E0_eV':float(energies[0]),'analytic_E0_eV':expected,'residual':float(residual)})
    assert abs(energies[0]-expected)<1e-9 and residual<1e-9
else:
    print('Optional Clifford QC benchmark skipped (not needed for one-body reproduction).')

## 8. Download private results, then review before sharing

Colab runtimes are ephemeral. The following optional download contains output only; it does **not** copy any raw production `.dat` to GitHub. The archive should be shared only with people authorized to access the original data.

In [ ]:
MAKE_DOWNLOAD = False
if MAKE_DOWNLOAD:
    import shutil
    archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
    print('Archive:',archive)
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        print('Download this archive manually:',archive)